In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

CANDIDATE_BATCH = "group-5-batch-5"
MAX_ROWS = 75_000

db_url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:"
    f"{os.environ['POSTGRES_PASSWORD']}@postgres:5432/"
    f"{os.environ['POSTGRES_DB']}"
)
engine = create_engine(db_url)

candidate_data = pd.read_sql(
    """
    SELECT
        brokered_by, status, bed, bath, acre_lot, street,
        city, state, zip_code, house_size, prev_sold_year, target_price
    FROM gold.property_training_features
    WHERE batch_id = %(batch_id)s
      AND target_price IS NOT NULL
      AND target_price > 0
    LIMIT %(limit)s
    """,
    engine,
    params={"batch_id": CANDIDATE_BATCH, "limit": MAX_ROWS},
)

print("Filas para candidate:", len(candidate_data))
candidate_data["target_price"].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99, 0.995, 0.999]
)

Filas para candidate: 75000


count    7.500000e+04
mean     1.218523e+06
std      2.109273e+06
min      5.002300e+05
1%       5.100000e+05
5%       5.250000e+05
50%      7.499990e+05
95%      3.095000e+06
99%      7.995000e+06
99.5%    1.250000e+07
99.9%    2.650000e+07
max      1.500000e+08
Name: target_price, dtype: float64

In [2]:
import mlflow
import mlflow.sklearn
import numpy as np
from mlflow import MlflowClient
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

FEATURE_COLUMNS = [
    "brokered_by", "bed", "bath", "acre_lot", "street",
    "zip_code", "house_size", "prev_sold_year",
    "status", "city", "state",
]
NUMERIC_COLUMNS = FEATURE_COLUMNS[:8]
CATEGORICAL_COLUMNS = ["status", "city", "state"]

X_train, X_test, y_train, y_test = train_test_split(
    candidate_data[FEATURE_COLUMNS],
    candidate_data["target_price"],
    test_size=0.20,
    random_state=42,
)

def build_candidate(log_target=False):
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), NUMERIC_COLUMNS),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            )),
        ]), CATEGORICAL_COLUMNS),
    ])

    regressor = HistGradientBoostingRegressor(
        max_iter=200,
        max_leaf_nodes=63,
        l2_regularization=0.5,
        random_state=42,
    )

    if log_target:
        regressor = TransformedTargetRegressor(
            regressor=regressor,
            func=np.log1p,
            inverse_func=np.expm1,
        )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", regressor),
    ])

def score_model(name, model):
    prediction = model.predict(X_test)
    return {
        "model": name,
        "mae": mean_absolute_error(y_test, prediction),
        "rmse": np.sqrt(mean_squared_error(y_test, prediction)),
        "r2": r2_score(y_test, prediction),
    }

mlflow.set_tracking_uri("http://mlflow:5000")
champion = mlflow.sklearn.load_model("models:/property_price_model@champion")

raw_candidate = build_candidate(log_target=False).fit(X_train, y_train)
log_candidate = build_candidate(log_target=True).fit(X_train, y_train)

candidate_comparison = pd.DataFrame([
    score_model("champion_v1", champion),
    score_model("candidate_raw_target", raw_candidate),
    score_model("candidate_log_target", log_candidate),
]).sort_values("mae")

candidate_comparison

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,mae,rmse,r2
2,candidate_log_target,394400.966928,1.617979e+06,0.585207
1,candidate_raw_target,452951.294022,1.571913e+06,0.608490
0,champion_v1,842152.435903,2.646055e+06,-0.109387


In [4]:
REFERENCE_BATCH = "group-5-batch-4"
reference_eval = pd.read_sql(
    """
    SELECT
        brokered_by, status, bed, bath, acre_lot, street,
        city, state, zip_code, house_size, prev_sold_year, target_price
    FROM gold.property_training_features
    WHERE batch_id = %(batch_id)s
      AND target_price IS NOT NULL
      AND target_price > 0
    LIMIT 10000
    """,
    engine,
    params={"batch_id": REFERENCE_BATCH},
)

def score_external(name, model, data):
    prediction = model.predict(data[FEATURE_COLUMNS])
    actual = data["target_price"]
    return {
        "model": name,
        "mae": mean_absolute_error(actual, prediction),
        "rmse": np.sqrt(mean_squared_error(actual, prediction)),
        "r2": r2_score(actual, prediction),
    }

batch4_safety_comparison = pd.DataFrame([
    score_external("champion_v1", champion, reference_eval),
    score_external("candidate_log_target", log_candidate, reference_eval),
]).sort_values("mae")

batch4_safety_comparison

,model,mae,rmse,r2
0,champion_v1,40677.616813,48774.126203,0.283242
1,candidate_log_target,382941.811331,426897.126111,-53.908573


In [5]:
BATCH4_TRAIN_ROWS = 25_000
BATCH5_TRAIN_ROWS = 50_000

batch4_candidate_data = pd.read_sql(
    """
    SELECT
        brokered_by, status, bed, bath, acre_lot, street,
        city, state, zip_code, house_size, prev_sold_year,
        target_price, batch_id
    FROM gold.property_training_features
    WHERE batch_id = 'group-5-batch-4'
      AND target_price IS NOT NULL
      AND target_price > 0
    LIMIT %(limit)s
    """,
    engine,
    params={"limit": BATCH4_TRAIN_ROWS},
)

batch5_candidate_data = candidate_data.copy()
batch5_candidate_data["batch_id"] = CANDIDATE_BATCH
batch5_candidate_data = batch5_candidate_data.sample(
    n=BATCH5_TRAIN_ROWS,
    random_state=42,
)

blended_data = pd.concat(
    [batch4_candidate_data, batch5_candidate_data],
    ignore_index=True,
)

print(blended_data["batch_id"].value_counts())

batch_id
group-5-batch-5    50000
group-5-batch-4    25000
Name: count, dtype: int64


In [6]:
X_train, X_holdout, y_train, y_holdout, batch_train, batch_holdout = train_test_split(
    blended_data[FEATURE_COLUMNS],
    blended_data["target_price"],
    blended_data["batch_id"],
    test_size=0.20,
    random_state=42,
    stratify=blended_data["batch_id"],
)

blended_candidate = build_candidate(log_target=True)
blended_candidate.fit(X_train, y_train)

def score_by_batch(name, model):
    results = []

    for batch_id in sorted(batch_holdout.unique()):
        mask = batch_holdout == batch_id
        actual = y_holdout[mask]
        prediction = model.predict(X_holdout.loc[mask])

        results.append({
            "model": name,
            "batch_id": batch_id,
            "mae": mean_absolute_error(actual, prediction),
            "rmse": np.sqrt(mean_squared_error(actual, prediction)),
            "r2": r2_score(actual, prediction),
        })

    return results

blended_comparison = pd.DataFrame(
    score_by_batch("champion_v1", champion)
    + score_by_batch("candidate_blended_log", blended_candidate)
).sort_values(["batch_id", "mae"])

blended_comparison

,model,batch_id,mae,rmse,r2
0,champion_v1,group-5-batch-4,40520.418821,4.843433e+04,0.278119
2,candidate_blended_log,group-5-batch-4,63721.558307,9.795224e+04,-1.952484
3,candidate_blended_log,group-5-batch-5,381489.086328,1.204519e+06,0.594326
1,champion_v1,group-5-batch-5,781618.320955,2.041891e+06,-0.165776


In [7]:
batch4_balanced = pd.read_sql(
    """
    SELECT
        brokered_by, status, bed, bath, acre_lot, street,
        city, state, zip_code, house_size, prev_sold_year,
        target_price, batch_id
    FROM gold.property_training_features
    WHERE batch_id = 'group-5-batch-4'
      AND target_price IS NOT NULL
      AND target_price > 0
    LIMIT 37500
    """,
    engine,
)

batch5_balanced = candidate_data.sample(n=37500, random_state=42).copy()
batch5_balanced["batch_id"] = CANDIDATE_BATCH

balanced_data = pd.concat(
    [batch4_balanced, batch5_balanced],
    ignore_index=True,
)

balanced_data["batch_id"].value_counts()

batch_id
group-5-batch-4    37500
group-5-batch-5    37500
Name: count, dtype: int64

In [8]:
X_train, X_holdout, y_train, y_holdout, batch_train, batch_holdout = train_test_split(
    balanced_data[FEATURE_COLUMNS],
    balanced_data["target_price"],
    balanced_data["batch_id"],
    test_size=0.20,
    random_state=42,
    stratify=balanced_data["batch_id"],
)

balanced_candidate = build_candidate(log_target=True)
balanced_candidate.fit(X_train, y_train)

balanced_comparison = pd.DataFrame(
    score_by_batch("champion_v1", champion)
    + score_by_batch("candidate_balanced_log", balanced_candidate)
).sort_values(["batch_id", "mae"])

balanced_comparison

,model,batch_id,mae,rmse,r2
0,champion_v1,group-5-batch-4,40202.730349,4.850197e+04,0.273760
2,candidate_balanced_log,group-5-batch-4,59841.526579,1.008463e+05,-2.139649
3,candidate_balanced_log,group-5-batch-5,423865.288059,1.564014e+06,0.525757
1,champion_v1,group-5-batch-5,819332.736171,2.410726e+06,-0.126717


In [9]:
balanced_raw_candidate = build_candidate(log_target=False)
balanced_raw_candidate.fit(X_train, y_train)

balanced_raw_comparison = pd.DataFrame(
    score_by_batch("champion_v1", champion)
    + score_by_batch("candidate_balanced_raw", balanced_raw_candidate)
).sort_values(["batch_id", "mae"])

balanced_raw_comparison

,model,batch_id,mae,rmse,r2
0,champion_v1,group-5-batch-4,40202.730349,4.850197e+04,0.273760
2,candidate_balanced_raw,group-5-batch-4,99133.057220,1.749373e+05,-8.447695
3,candidate_balanced_raw,group-5-batch-5,485110.144029,1.566613e+06,0.524180
1,champion_v1,group-5-batch-5,819332.736171,2.410726e+06,-0.126717


In [14]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

NUMERIC_FEATURES = [
    "bed", "bath", "acre_lot", "house_size", "prev_sold_year",
]

def population_stability_index(reference, current, bins=10):
    reference = reference.dropna()
    current = current.dropna()

    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return np.nan

    edges[0], edges[-1] = -np.inf, np.inf
    intervals = pd.cut(reference, bins=edges, include_lowest=True).cat.categories

    ref_share = (
        pd.cut(reference, bins=edges, include_lowest=True)
        .value_counts()
        .reindex(intervals, fill_value=0) / len(reference)
    )
    cur_share = (
        pd.cut(current, bins=edges, include_lowest=True)
        .value_counts()
        .reindex(intervals, fill_value=0) / len(current)
    )

    epsilon = 1e-6
    return float(
        ((cur_share - ref_share) *
         np.log((cur_share + epsilon) / (ref_share + epsilon))).sum()
    )

rows = []
for feature in NUMERIC_FEATURES:
    ks_stat, ks_pvalue = ks_2samp(
        reference_df[feature].dropna(),
        current_df[feature].dropna(),
    )
    psi = population_stability_index(
        reference_df[feature],
        current_df[feature],
    )

    rows.append({
        "feature": feature,
        "psi": round(psi, 4),
        "ks_statistic": round(float(ks_stat), 4),
        "ks_pvalue": float(ks_pvalue),
        "drift_detected": bool(psi >= 0.25 or ks_stat >= 0.10),
    })

numeric_drift_df = pd.DataFrame(rows).sort_values("psi", ascending=False)
numeric_drift_df

,feature,psi,ks_statistic,ks_pvalue,drift_detected
3,house_size,0.4449,0.2721,0.000000e+00,True
1,bath,0.3951,0.2385,7.602229e-254,True
0,bed,0.2213,0.1925,1.025351e-164,True
2,acre_lot,0.0379,0.0728,9.447698e-24,False
4,prev_sold_year,0.0175,0.0530,8.691649e-13,False


In [15]:
CATEGORICAL_FEATURES = ["zip_code", "status", "city", "state"]

def categorical_metrics(reference, current):
    reference = reference.fillna("__MISSING__").astype(str)
    current = current.fillna("__MISSING__").astype(str)

    ref_share = reference.value_counts(normalize=True)
    cur_share = current.value_counts(normalize=True)
    categories = ref_share.index.union(cur_share.index)

    ref_share = ref_share.reindex(categories, fill_value=0)
    cur_share = cur_share.reindex(categories, fill_value=0)

    epsilon = 1e-6
    psi = float(
        ((cur_share - ref_share) *
         np.log((cur_share + epsilon) / (ref_share + epsilon))).sum()
    )
    tvd = float(0.5 * (ref_share - cur_share).abs().sum())
    new_rate = float((~current.isin(set(reference))).mean())

    return psi, tvd, new_rate

rows = []
for feature in CATEGORICAL_FEATURES:
    psi, tvd, new_rate = categorical_metrics(
        reference_df[feature],
        current_df[feature],
    )
    rows.append({
        "feature": feature,
        "psi": round(psi, 4),
        "distribution_change": round(tvd, 4),
        "new_category_rate": round(new_rate, 4),
        "drift_detected": bool(psi >= 0.25 or tvd >= 0.20),
    })

categorical_drift_df = pd.DataFrame(rows).sort_values(
    "distribution_change",
    ascending=False,
)
categorical_drift_df

,feature,psi,distribution_change,new_category_rate,drift_detected
0,zip_code,5.9877,0.6945,0.5235,True
2,city,3.8307,0.5616,0.2835,True
3,state,0.5192,0.3048,0.0004,True
1,status,0.0026,0.0251,0.0000,False


In [16]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RIDGE_NUMERIC = [
    "bed", "bath", "acre_lot",
    "house_size", "prev_sold_year",
]
RIDGE_CATEGORICAL = ["zip_code", "status", "city", "state"]

ridge_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), RIDGE_NUMERIC),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), RIDGE_CATEGORICAL),
], remainder="drop")

ridge_candidate = Pipeline([
    ("preprocessor", ridge_preprocessor),
    ("regressor", TransformedTargetRegressor(
        regressor=Ridge(alpha=10.0, solver="lsqr"),
        func=np.log1p,
        inverse_func=np.expm1,
    )),
])

ridge_candidate.fit(X_train, y_train)

ridge_comparison = pd.DataFrame(
    score_by_batch("champion_v1", champion)
    + score_by_batch("candidate_ridge_log", ridge_candidate)
).sort_values(["batch_id", "mae"])

ridge_comparison

,model,batch_id,mae,rmse,r2
0,champion_v1,group-5-batch-4,4.020273e+04,4.850197e+04,2.737603e-01
2,candidate_ridge_log,group-5-batch-4,7.285718e+04,1.287899e+05,-4.120647e+00
1,champion_v1,group-5-batch-5,8.193327e+05,2.410726e+06,-1.267171e-01
3,candidate_ridge_log,group-5-batch-5,4.453269e+07,3.798557e+09,-2.797412e+06
